# Stage 1: Non-Instruction Fine-Tuning
## Domain: E-commerce Product Support Assistant

**Goal:** Adapt the base model (`Qwen2.5-0.5B`) to e-commerce support domain language and
terminology by training on raw, unstructured support text (next-token prediction only,
no instruction/response format). This is the first stage of a 3-stage pipeline:

`Base Model -> Stage 1 (Non-Instruction FT) -> Stage 2 (Instruction FT) -> Stage 3 (DPO)`


## 1. Install required libraries

In [1]:
%%capture
import os
!pip install unsloth
!pip install --upgrade --no-cache-dir "unsloth[colab-new]"
!pip install trl peft accelerate bitsandbytes


## 2. Load raw domain text

In [2]:
raw_text_path = "/content/non_instruction_data.txt"

# In Colab, upload data/non_instruction_data.txt from this repo, or mount Drive.
# from google.colab import files
# files.upload()

with open(raw_text_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Total characters:", len(raw_text))
print(raw_text[:500])


Total characters: 21015
Customers often ask whether an order can be cancelled after payment. Sometimes the warehouse
has already packed it, sometimes not. The exact timing changes the outcome and support notes
are not always perfectly formatted. Internal note 1. Sometimes punctuation missing sometimes
extra spaces customer typed okay.. support replied please wait 24-48 hrs and recheck. Inventory
sync maybe delayed, address validation maybe failed, nothing guaranteed until final scan.
Return requests arrive for clothing


## 3. Clean and chunk the text

The raw PDF text contains repeated boilerplate ("Internal note N...") and inconsistent
punctuation/spacing (typical of real scraped support logs). We do light cleaning, then
split into paragraph-level chunks suitable for next-token-prediction training.

In [3]:
import re

def clean_text(text):
    text = re.sub(r"\s+", " ", text)            # collapse whitespace
    text = re.sub(r"\.\.+", ".", text)            # fix repeated periods
    text = text.strip()
    return text

# Split into paragraphs using sentence groups (each "topic" paragraph in the raw PDF)
raw_paragraphs = re.split(r"(?=Customers often ask|Return requests arrive|Delivery tracking|"
                           r"Gift cards usually|A damaged package|Electronic products occasionally|"
                           r"Marketplace sellers maintain|Refund processing depends|"
                           r"Customers forget passwords|Subscription products renew)", raw_text)

chunks = []
for p in raw_paragraphs:
    p = clean_text(p)
    if len(p) > 80:           # drop tiny/empty fragments
        # remove "Internal note N" boilerplate lines
        p = re.sub(r"Internal note \d+\.", "", p)
        chunks.append(p.strip())

print("Number of chunks:", len(chunks))
print(chunks[0][:400])


Number of chunks: 50
Customers often ask whether an order can be cancelled after payment. Sometimes the warehouse has already packed it, sometimes not. The exact timing changes the outcome and support notes are not always perfectly formatted.  Sometimes punctuation missing sometimes extra spaces customer typed okay. support replied please wait 24-48 hrs and recheck. Inventory sync maybe delayed, address validation may


In [4]:
import json

with open("/content/non_instruction_chunks.jsonl", "w") as f:
    for c in chunks:
        f.write(json.dumps({"text": c}) + "\n")

print("Saved", len(chunks), "chunks to non_instruction_chunks.jsonl")


Saved 50 chunks to non_instruction_chunks.jsonl


## 4. Load base model using Unsloth

In [5]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 1024
BASE_MODEL = "unsloth/Qwen2.5-0.5B-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = True,
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/457M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

unsloth/Qwen2.5-0.5B-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


## 5. Apply LoRA

In [6]:
STAGE1_RANK = 16
STAGE1_ALPHA = 16
STAGE1_DROPOUT = 0.0
STAGE1_LR = 2e-4
STAGE1_MAX_STEPS = 60
STAGE1_BATCH_SIZE = 2

model = FastLanguageModel.get_peft_model(
    model,
    r = STAGE1_RANK,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha = STAGE1_ALPHA,
    lora_dropout = STAGE1_DROPOUT,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)


Unsloth 2026.6.9 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


## 6. Train on raw text (next-token prediction, no instruction format)

In [7]:
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

dataset = load_dataset("json", data_files="/content/non_instruction_chunks.jsonl", split="train")
print(dataset)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    packing = True,                # short raw paragraphs -> pack into sequences
    args = SFTConfig(
        per_device_train_batch_size = STAGE1_BATCH_SIZE,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = STAGE1_MAX_STEPS,
        learning_rate = STAGE1_LR,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "/content/stage1_outputs",
    ),
)

stage1_stats = trainer.train()
print(stage1_stats)


Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 50
})


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/50 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 50 | Num Epochs = 9 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
5,5.430643
10,3.853000
15,2.342709
20,1.405518
25,0.886173
30,0.601989
35,0.293926
40,0.127757
45,0.050832
50,0.038132


Unsloth: Restored added_tokens_decoder metadata in /content/stage1_outputs/checkpoint-60/tokenizer_config.json.


TrainOutput(global_step=60, training_loss=1.257172809789578, metrics={'train_runtime': 99.3757, 'train_samples_per_second': 4.83, 'train_steps_per_second': 0.604, 'total_flos': 67266039548160.0, 'train_loss': 1.257172809789578, 'epoch': 8.64})


## 7. Save the Stage 1 adapter / model

In [8]:
STAGE1_ADAPTER_DIR = "/content/stage1_non_instruction_adapter"
model.save_pretrained(STAGE1_ADAPTER_DIR)
tokenizer.save_pretrained(STAGE1_ADAPTER_DIR)
print("Saved Stage 1 adapter to:", STAGE1_ADAPTER_DIR)

# Optionally merge to 16-bit for use as the Stage 2 starting checkpoint
STAGE1_MERGED_DIR = "/content/stage1_non_instruction_merged_model"
model.save_pretrained_merged(STAGE1_MERGED_DIR, tokenizer, save_method = "merged_16bit")
print("Saved Stage 1 merged model to:", STAGE1_MERGED_DIR)


Unsloth: Restored added_tokens_decoder metadata in /content/stage1_non_instruction_adapter/tokenizer_config.json.


Saved Stage 1 adapter to: /content/stage1_non_instruction_adapter


config.json:   0%|          | 0.00/774 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /content/stage1_non_instruction_merged_model/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:14<00:00, 14.40s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:07<00:00,  7.58s/it]


Unsloth: Merge process complete. Saved to `/content/stage1_non_instruction_merged_model`
Saved Stage 1 merged model to: /content/stage1_non_instruction_merged_model


## 8. Test the model after non-instruction fine-tuning

In [9]:
FastLanguageModel.for_inference(model)

test_prompt = "Customers often ask whether an order can be cancelled after payment."

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=80, use_cache=True)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Both `max_new_tokens` (=80) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Customers often ask whether an order can be cancelled after payment. Sometimes the warehouse has already packed it, sometimes not. The exact timing changes the outcome and support notes are not always perfectly formatted.  Sometimes punctuation missing sometimes extra spaces customer typed okay. support replied please wait 24-48 hrs and recheck. Inventory sync maybe delayed, address validation maybe failed, nothing guaranteed until final scan. Customer downloads order and looks it over before finalizing order.


### Observation
After Stage 1, the model should produce text that "sounds like" e-commerce support
language (talking about warehouses, packing, refunds, tracking, etc.) even though it has
not yet learned to answer direct questions in an instruction/response format. That
behavior is added in Stage 2.